In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

DataFrame[]

In [0]:
from pyspark.sql.functions import current_timestamp

caminho_base = "/Volumes/workspace/default/inputs/"

def ingerir_csv(nome_arquivo, nome_tabela):
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(caminho_base + nome_arquivo))
    df_bronze = df.withColumn("ingestion_datetime", current_timestamp())
    (df_bronze.write
     .format("delta")
     .mode("append")
     .saveAsTable(nome_tabela))
    
    print(f"{nome_tabela}: {df_bronze.count()} linhas gravadas")

In [0]:
ingerir_csv("movies_info_TMDB_IMDB.csv", "bronze.tb_movies_info")
ingerir_csv("movies_financials_IMDB_TMDB.csv", "bronze.tb_movies_financials")
ingerir_csv("movies_metrics_IMDB_TMDB.csv", "bronze.tb_movies_metrics")
ingerir_csv("credits_and_tags_IMDB_TMDB.csv", "bronze.tb_credits_and_tags")
ingerir_csv("movies_reviews.csv", "bronze.tb_movies_reviews")

bronze.tb_movies_info: 106930 linhas gravadas
bronze.tb_movies_financials: 106165 linhas gravadas
bronze.tb_movies_metrics: 107364 linhas gravadas
bronze.tb_credits_and_tags: 106320 linhas gravadas
bronze.tb_movies_reviews: 32412 linhas gravadas


In [0]:
import requests
from datetime import datetime, timedelta

#  parâmetros (widgets) do notebook para as datas de início e fim
dbutils.widgets.text("data_fim", datetime.today().strftime("%m-%d-%Y"))
dbutils.widgets.text("data_inicio", (datetime.today() - timedelta(days=7)).strftime("%m-%d-%Y"))

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url)
dados = resposta.json()["value"]

df_cotacao = spark.createDataFrame(dados).withColumn("ingestion_datetime", current_timestamp())
df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
display(df_cotacao)


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.1143,2026-09-10 13:09:28.70012,2026-09-21T12:42:13.492Z
5.0912,2026-09-11 13:07:22.532196,2026-09-21T12:42:13.492Z
5.169,2026-09-14 13:10:08.144425,2026-09-21T12:42:13.492Z
5.1484,2026-09-15 13:09:19.199664,2026-09-21T12:42:13.492Z
5.152,2026-09-16 13:05:30.35873,2026-09-21T12:42:13.492Z
5.1515,2026-09-17 13:03:21.858212,2026-09-21T12:42:13.492Z


In [0]:
%sql
SELECT * FROM bronze.tb_cotacao_dolar

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.1143,2026-09-10 13:09:28.70012,2026-09-21T12:42:12.323Z
5.0912,2026-09-11 13:07:22.532196,2026-09-21T12:42:12.323Z
5.169,2026-09-14 13:10:08.144425,2026-09-21T12:42:12.323Z
5.1484,2026-09-15 13:09:19.199664,2026-09-21T12:42:12.323Z
5.152,2026-09-16 13:05:30.35873,2026-09-21T12:42:12.323Z
5.1515,2026-09-17 13:03:21.858212,2026-09-21T12:42:12.323Z
5.1143,2026-09-10 13:09:28.70012,2026-09-17T18:52:47.750Z
5.0912,2026-09-11 13:07:22.532196,2026-09-17T18:52:47.750Z
5.169,2026-09-14 13:10:08.144425,2026-09-17T18:52:47.750Z
5.1484,2026-09-15 13:09:19.199664,2026-09-17T18:52:47.750Z


In [0]:
%sql
SELECT 'tb_movies_info' AS tabela, COUNT(*) AS qtd_linhas FROM bronze.tb_movies_info
UNION ALL
SELECT 'tb_movies_financials', COUNT(*) FROM bronze.tb_movies_financials
UNION ALL
SELECT 'tb_movies_metrics', COUNT(*) FROM bronze.tb_movies_metrics
UNION ALL
SELECT 'tb_credits_and_tags', COUNT(*) FROM bronze.tb_credits_and_tags
UNION ALL
SELECT 'tb_movies_reviews', COUNT(*) FROM bronze.tb_movies_reviews
UNION ALL
SELECT 'tb_cotacao_dolar', COUNT(*) FROM bronze.tb_cotacao_dolar

tabela,qtd_linhas
tb_movies_info,213860
tb_movies_financials,212330
tb_movies_metrics,214728
tb_credits_and_tags,212640
tb_movies_reviews,64824
tb_cotacao_dolar,12


In [0]:
%sql
SELECT * FROM bronze.tb_credits_and_tags LIMIT 10

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-17T18:44:14.798Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-17T18:44:14.798Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-17T18:44:14.798Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-17T18:44:14.798Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-17T18:44:14.798Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-09-17T18:44:14.798Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-09-17T18:44:14.798Z
315635,"Action, Adventure, Science Fiction, Drama","Ma